# Lab 0 Assignment: Verify Your Environment With Code

Use this notebook after you finish `03_environment_check.ipynb`.

This assignment asks you to run and complete a few short code tasks so you can prove your local setup is working.

You will:
- locate the repo root in code
- load `.env` in code
- contact the configured Ollama endpoint in code
- confirm the default model is available
- ask the configured model one question
- build a short environment summary in Python
- observe what your endpoint returns and report it clearly

In [1]:
import json
from pathlib import Path

import requests
from dotenv import dotenv_values
from openai import OpenAI

# Teaching note:
# This setup cell loads the libraries used in the assignment, assumes you opened the
# notebook from this lab folder, and then locates this lab's local .env file.
LAB_NAME = 'lab0_1_environment_setup'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(
        f'Open this notebook from the {LAB_NAME} folder.'
    )

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError(
        f'Expected .env in this folder. Copy .env.example to .env first.'
    )

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)


Repo root: /home/frank/projects/agentic-AI4-forensics
Lab folder: /home/frank/projects/agentic-AI4-forensics/lab0_1_environment_setup


## Task 1

Run the cell below and confirm that the three required repo paths exist.

Expected result:
- `.env` exists
- `requirements.txt` exists
- `src/` exists

In [2]:
# Teaching note:
# This block builds the key repo and lab paths used in the rest of the assignment.
# In a working setup, each printed check should return True.
requirements_path = repo_root / 'requirements.txt'
src_path = repo_root / 'src'

print('.env exists:', env_path.exists())
print('requirements.txt exists:', requirements_path.exists())
print('src exists:', src_path.exists())


.env exists: True
requirements.txt exists: True
src exists: True


## Task 2

Load `.env` and print the configured model and endpoint.

In [3]:
# Teaching note:
# This block reads this lab's .env and pulls out the model name and Ollama URL used later.
# It stops early if either required setting is missing.
config = dotenv_values(env_path)
model = config.get('MODEL')
ollama_base_url = config.get('OLLAMA_BASE_URL')

print('MODEL:', model)
print('OLLAMA_BASE_URL:', ollama_base_url)

if not model or not ollama_base_url:
    raise ValueError("MODEL or OLLAMA_BASE_URL is missing from this lab's .env")


MODEL: qwen3.5:9b
OLLAMA_BASE_URL: http://localhost:11434/v1


## Task 3

Build the `/api/tags` URL from `OLLAMA_BASE_URL`, send the request, and list the available models.

In [4]:
# Teaching note:
# This block turns the OpenAI-compatible /v1 URL into Ollama's /api/tags endpoint.
# It then asks the server for the available model list.
api_root = ollama_base_url.rstrip('/')
tags_url = api_root.removesuffix('/v1') + '/api/tags'

response = requests.get(tags_url, timeout=10)
response.raise_for_status()
available_models = [item.get('name') for item in response.json().get('models', []) if item.get('name')]

print('Tags URL:', tags_url)
print('Available models:')
for model_name in available_models:
    print('-', model_name)


Tags URL: http://localhost:11434/api/tags
Available models:
- gemma4:31b
- gemma4:e4b
- huihui_ai/qwen3.5-abliterated:latest
- qwen3.5:2b
- qwen3.5:9b
- qwen3.5:0.8b
- qwen3.5:35b-a3b
- qwen3:8b
- qwen3.5:27b
- llama3.3:70b


## Task 4

Check whether the default model from `.env` is available at the configured endpoint.

In [5]:
# Teaching note:
# This block checks whether the default model from .env is actually available.
# That is the minimum model check needed before the next lab.
model_available = model in available_models

print('Default model available:', model_available)

if not model_available:
    raise ValueError(f'The model {model!r} is not available at the configured endpoint.')


Default model available: True


## Task 5

Ask the configured model one question.

In a chat request, each message has a `role`. In this first example, you only use the `user` role because that is the role for the actual question or request. Later labs will also use the `system` role to give the model higher-level instructions about how it should behave.

Replace the example text in `student_question` with one question of your own, then run the cell.

In [6]:
# Teaching note:
# Replace the example question below, then send one request to the configured model.
# This confirms the notebook can use the same endpoint and model settings as later labs.
# This first example uses only a user message. Later labs may add a system message
# to give the model higher-level behavior instructions.
student_question = 'What is one good habit for organizing digital evidence files?'

client = OpenAI(base_url=ollama_base_url, api_key='ollama')
response = client.chat.completions.create(
    model=model,
    messages=[{'role': 'user', 'content': student_question}],
)

model_answer = response.choices[0].message.content.strip()

print('Question:', student_question)
print('\nModel answer:')
print(model_answer)


Question: What is one good habit for organizing digital evidence files?

Model answer:
One of the most critical habits for organizing digital evidence files is:

**Maintain a strict separation between raw evidence and analysis work products.**

### Why this habit is essential:
1.  **Preserves Integrity:** By keeping the original, unmodified evidence (usually stored in a read-only, archived format) in a separate folder structure from your notes, code, and processed files, you demonstrate that the evidence has not been tampered with.
2.  **Ensures Defensibility:** In a legal or investigative context, this separation creates a clear audit trail. It proves that your findings are based on the original data, not on manipulated versions created during your work.
3.  **Facilitates Recovery:** If you make a mistake in your analysis, you can revert to the source without affecting the original file.

**How to implement it:**
*   Create a directory for **Raw_Evidence** that is set to Read-Only.
* 

## Task 6

Run the cell below and review the environment summary as formatted JSON.

For `repo_root_name`, use only the folder name, not the full absolute path.

In [7]:
# Teaching note:
# This block collects the main setup results into one small summary dictionary.
# The summary makes it easier to review readiness before moving on.
environment_summary = {
    'repo_root_name': repo_root.name,
    'default_model': model,
    'ollama_base_url': ollama_base_url,
    'available_model_count': len(available_models),
    'default_model_available': model_available,
}

print(json.dumps(environment_summary, indent=2))


{
  "repo_root_name": "agentic-AI4-forensics",
  "default_model": "qwen3.5:9b",
  "ollama_base_url": "http://localhost:11434/v1",
  "available_model_count": 10,
  "default_model_available": true
}


## Task 7: Observation Report

This is a short write-up task based on the results you already printed in Tasks 3 through 6.

You do not need to contact the endpoint again. Just look at your earlier outputs and fill in the report below.

Type your answers by replacing the five empty strings in the `setup_report` code cell below.

Include:
- how many models were returned in the printed model list from Task 3
- whether the default model from `.env` was available
- one other model name you noticed in the list
- whether your environment is ready for `lab0_2_model_warmup`
- a short 1-2 sentence explanation of your answer

In [8]:
# Teaching note:
# Fill in this short report using your outputs from Tasks 3 through 5.
# Replace each empty string with a plain-language answer before submitting.
setup_report = {
    'observed_model_count': '',
    'default_model_status': '',
    'one_other_model_you_noticed': '',
    'ready_for_lab0_2_model_warmup': '',
    'readiness_explanation': '',
}

# Teaching note:
# This check points out any report fields that are still blank before submission.
required_text_fields = [
    'observed_model_count',
    'default_model_status',
    'one_other_model_you_noticed',
    'ready_for_lab0_2_model_warmup',
    'readiness_explanation',
]

missing = [key for key in required_text_fields if not str(setup_report[key]).strip()]

print(json.dumps(setup_report, indent=2))

if missing:
    print(f"\nPlease complete these fields before submitting: {', '.join(missing)}")
else:
    print("\nObservation report complete.")


{
  "observed_model_count": "",
  "default_model_status": "",
  "one_other_model_you_noticed": "",
  "ready_for_lab0_2_model_warmup": "",
  "readiness_explanation": ""
}

Please complete these fields before submitting: observed_model_count, default_model_status, one_other_model_you_noticed, ready_for_lab0_2_model_warmup, readiness_explanation


## Submission

Save the notebook with:
- the completed code cells
- the printed model list
- the printed question and model answer from Task 5
- the final `environment_summary` output
- the completed `setup_report` observation report